In [ ]:

# -- Cell 1 -- rclone + Drive. Same pattern as the earlier notebooks.
# Requires: Settings -> Internet ON, Accelerator GPU T4, RCLONE_DRIVE_TOKEN
# attached to THIS notebook.
import os, subprocess

r = subprocess.run("curl -s https://rclone.org/install.sh | sudo bash", shell=True)
if r.returncode not in (0, 3):
    raise RuntimeError("rclone install failed (exit %d)" % r.returncode)

from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("RCLONE_DRIVE_TOKEN")
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write("[drive]\ntype = drive\nscope = drive\ntoken = " + token + "\n")

REMOTE = "drive:Distillation"
out = subprocess.run("rclone lsf " + REMOTE, shell=True, capture_output=True, text=True)
print(out.stdout or out.stderr)
assert out.returncode == 0, "cannot see " + REMOTE


In [ ]:

# -- Cell 2 -- deps + GPU.
#
# This run uses ONE GPU by design. The teacher and student live in the same
# process and the same card; there is no split-device or DDP path here. A second
# GPU, if the session has one, simply sits idle -- see the note in Cell 6 about
# what that costs.
subprocess.run('pip install -q -U "transformers>=5.0" pyarrow', shell=True, check=True)
subprocess.run("pip uninstall -y -q torchao", shell=True)   # peft/torchao clash guard

import torch, numpy as np, pandas as pd, glob, json, time, signal
print("torch", torch.__version__, "| GPUs", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print("   cuda:%d %s %.0f GB" % (i, p.name, p.total_memory / 1e9))
assert torch.cuda.device_count() >= 1, "need a GPU"
if torch.cuda.device_count() > 1:
    print("\nNOTE: %d GPUs visible, only cuda:0 will be used."
          % torch.cuda.device_count())


In [ ]:

# -- Cell 3 -- pull the three inputs.
#
#   distill/       our training code                    (small)
#   subset parquet 2M molecules                         (~0.6 GB)
#   teacher        their released 337M MLM              (~1.4 GB)
#   student init   their released 32M MLM               (~0.13 GB)
#
# NO TEACHER CACHE. That is the whole point of this run: the 15 GB of cached
# top-16 logits from the earlier pipeline is replaced by running the teacher live,
# which gives full 405-way targets instead of a truncated top-16, and a fresh mask
# every epoch instead of one fixed mask per molecule. The price is paid in compute
# rather than storage -- Cell 6 measures exactly how much.
WORK = "/kaggle/working"
CODE, SUBSET_D = WORK + "/distill", WORK + "/subset"
TEACH, INIT = WORK + "/models/peptideclm-2-mlm-large", WORK + "/models/peptideclm-2-mlm-small"

def pull(remote, local, extra=""):
    os.makedirs(local, exist_ok=True)
    subprocess.run("rclone copy %s/%s %s --transfers 8 %s -P" % (REMOTE, remote, local, extra),
                   shell=True, check=True)

def rlsf(path):
    r = subprocess.run("rclone lsf " + path, shell=True, capture_output=True, text=True)
    return r.stdout.split() if r.returncode == 0 else []

def pull_model(name, local):
    """The two models sit on Drive in DIFFERENT layouts, so try both.

      flat:     models/<name>/model.safetensors            (how the 32M was uploaded)
      HF cache: models/models--aaronfeller--<name>/snapshots/<sha>/   (how the 337M was)

    Guessing one layout is what broke here first time round: rclone lsf on a
    non-existent path exits 0 with empty output, so [0] raised IndexError instead
    of saying which path was missing."""
    if os.path.exists(local + "/model.safetensors"):
        return "already local"
    flat = "%s/models/%s" % (REMOTE, name)
    if any(f.startswith("model.safetensors") for f in rlsf(flat)):
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s %s -P" % (flat, local), shell=True, check=True)
        return "flat"
    snaps = "%s/models/models--aaronfeller--%s/snapshots" % (REMOTE, name)
    shas = [x.rstrip("/") for x in rlsf(snaps)]
    if shas:
        os.makedirs(local, exist_ok=True)
        subprocess.run("rclone copy %s/%s %s -P" % (snaps, shas[0], local),
                       shell=True, check=True)
        return "snapshot " + shas[0][:12]
    raise RuntimeError("%s not found on Drive. Tried  %s  and  %s"
                       % (name, flat, snaps))

if not os.path.exists(CODE + "/train_kd.py"):
    pull("distill", CODE)

# Prefer an attached Kaggle Dataset for the subset (instant), else Drive.
SUBSET = None
for cand in glob.glob("/kaggle/input/*/**/pretrain_subset_2M.parquet", recursive=True):
    SUBSET = cand; break
if SUBSET is None:
    if not glob.glob(SUBSET_D + "/*.parquet"):
        pull("data/pretrain_subset_2M", SUBSET_D, "--include '*.parquet'")
    SUBSET = glob.glob(SUBSET_D + "/*.parquet")[0]

print("teacher <- %s" % pull_model("peptideclm-2-mlm-large", TEACH))
print("student <- %s" % pull_model("peptideclm-2-mlm-small", INIT))
for d, who in ((TEACH, "teacher"), (INIT, "student")):
    for f in ("model.safetensors", "config.json"):
        assert os.path.exists(d + "/" + f), "%s is missing %s" % (who, f)

need = ["train_kd.py", "data_live.py", "losses_kd.py", "test_kd.py", "student.py"]
have = sorted(os.path.basename(f) for f in glob.glob(CODE + "/*.py"))
missing = [f for f in need if f not in have]
print("code    :", have)
assert not missing, "missing from drive:Distillation/distill -- %s" % missing
print("subset  : %s  %.2f GB" % (SUBSET, os.path.getsize(SUBSET) / 1e9))
print("teacher : %.2f GB" % (os.path.getsize(TEACH + "/model.safetensors") / 1e9))
print("student : %.2f GB" % (os.path.getsize(INIT + "/model.safetensors") / 1e9))


In [ ]:

# -- Cell 4 -- verify the inputs agree before spending hours.
#
# The failure this catches is specific to KD and completely silent: the soft loss
# is a KL between the teacher's and the student's distributions, so the two must
# be over the SAME vocabulary in the SAME order. If the checkpoints disagree --
# a different tokenizer revision, a resized head -- the KL is still a finite,
# plausible-looking number, and the run trains toward nonsense.
import sys
sys.path.insert(0, CODE)
from transformers import AutoTokenizer

ct = json.load(open(TEACH + "/config.json"))
cs = json.load(open(INIT + "/config.json"))
vt, vs = ct.get("vocab_size"), cs.get("vocab_size")
print("teacher: vocab %s  layers %s  hidden %s"
      % (vt, ct.get("num_hidden_layers"), ct.get("hidden_size")))
print("student: vocab %s  layers %s  hidden %s"
      % (vs, cs.get("num_hidden_layers"), cs.get("hidden_size")))
assert vt == vs, "vocab mismatch %s vs %s -- the KD term would be meaningless" % (vt, vs)

tok_t = AutoTokenizer.from_pretrained(TEACH, trust_remote_code=True)
tok_s = AutoTokenizer.from_pretrained(INIT, trust_remote_code=True)
meta = pd.read_parquet(SUBSET, columns=["source", "smiles", "n_tokens"])
probe = list(meta.smiles.iloc[:64])
ids_t = tok_t(probe, add_special_tokens=False)["input_ids"]
ids_s = tok_s(probe, add_special_tokens=False)["input_ids"]
assert ids_t == ids_s, "tokenizers disagree -- teacher and student would see different inputs"
for name in ("cls_token_id", "sep_token_id", "pad_token_id", "mask_token_id"):
    a, b = getattr(tok_t, name), getattr(tok_s, name)
    assert a == b and a is not None, "%s mismatch: %s vs %s" % (name, a, b)
print("tokenizers identical, special ids match, vocab %d" % vt)

# The tokenized length must match the cached n_tokens column, or the token-budget
# batching (and every time estimate below) is built on a wrong number.
#
# n_tokens ALREADY COUNTS [CLS] and [SEP] -- checked here rather than assumed.
# Getting that backwards adds 2 tokens per molecule to the epoch size, 1.3% of
# the corpus, and quietly skews every ETA below.
n_re = np.array([len(x) for x in ids_t]) + 2
assert (n_re == meta.n_tokens.iloc[:64].to_numpy()).all(), (
    "n_tokens does not equal tokenizer output + 2 specials -- stale column "
    "or the wrong tokenizer")

TOTAL_TOKENS = int(meta.n_tokens.sum())          # specials already included
print("\nmolecules {:,} | tokens/epoch {:,} (specials included)".format(len(meta), TOTAL_TOKENS))
print(meta.source.value_counts().to_string())
print("\ninputs are consistent")


In [ ]:

# -- Cell 5 -- unit tests on the real T4 and the real data.
# Everything below has already passed on an RTX 3050; this re-checks it here.
#
# The test that matters is the rotary-buffer one. Live teaching forces the 337M
# teacher and the 32M student into a single process, and that is exactly the
# configuration that silently corrupted their non-persistent RoPE buffers before
# -- it changed the student's outputs while the weights stayed byte-identical, and
# invalidated a round of geometry measurements. train_kd.py rebuilds and verifies
# those buffers on both models; the test confirms the student's outputs with the
# teacher resident are bit-identical to the student loaded alone.
r = subprocess.run(["python", "test_kd.py", "--subset", SUBSET, "--teacher", TEACH,
                    "--student", INIT, "--device", "cuda",
                    "--max-tokens", "4096", "--limit", "4000"],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-3000:])
assert r.returncode == 0, "tests failed -- do not start the real run"


In [ ]:

# -- Cell 6 -- measure throughput, then decide the epoch count. Do not skip this.
#
# WHY IT MATTERS. Compute here is dominated by the teacher, which is 10.5x the
# student and runs on every batch. Per token, forward is ~2N and backward ~4N:
#
#     teacher  337M, forward only      ~674 MFLOP/token   (78%)
#     student   32M, forward+backward  ~192 MFLOP/token   (22%)
#
# The earlier cache-building notebook is a measured anchor for the teacher half:
# 41 shards at ~24 min each = ~16 GPU-hours for THREE teacher passes over the
# corpus (clean + mask A + mask B), so roughly 5.5 GPU-hours per pass on a T4.
# One live epoch is one such pass plus the student, so ~7 h/epoch and ~14 h for
# two -- more than a single 12 h Kaggle session.
#
# That anchor is an extrapolation from a different notebook, so measure it instead
# of trusting it. The probe runs the real loop on real molecules for 40 steps.
#
# The probe is written INTO CODE, not into WORK. For `python <script>`, sys.path[0]
# is the SCRIPT's directory, not the cwd -- so a probe sitting in /kaggle/working
# cannot import data_live/losses_kd/train_kd no matter what cwd it is launched with.
PROBE = CODE + "/probe.py"
open(PROBE, "w").write(r"""
import argparse, sys, time, torch
ap = argparse.ArgumentParser()
ap.add_argument("--subset"); ap.add_argument("--teacher"); ap.add_argument("--student")
ap.add_argument("--max-tokens", type=int, default=8192)
ap.add_argument("--steps", type=int, default=40)
ap.add_argument("--warmup", type=int, default=8)
a = ap.parse_args()

from transformers import AutoTokenizer
from data_live import LiveStream
from losses_kd import hinton_kd_loss
from train_kd import load_pair, logits_of

tok = AutoTokenizer.from_pretrained(a.student, trust_remote_code=True)
teacher, student = load_pair(a.teacher, a.student, "cuda")
# The head 200k is representative: its source mix and mean length (158.9 tokens)
# match the full corpus (158.2), because the subset builder interleaved sources.
# Sampling the head would be wrong on a source-ordered file.
stream = LiveStream(a.subset, tok, max_tokens=a.max_tokens, seed=0,
                    val_molecules=0, split="train", limit=200_000)
opt = torch.optim.AdamW(student.parameters(), lr=1e-4)
scaler = torch.amp.GradScaler("cuda")
student.train()

used = pad = 0
t0 = None
for i, b in enumerate(stream.iter_epoch(0)):
    if i >= a.steps:
        break
    b = {k: v.to("cuda", non_blocking=True) for k, v in b.items()}
    with torch.autocast("cuda", dtype=torch.float16):
        with torch.no_grad():
            tl = logits_of(teacher(input_ids=b["input_ids"],
                                   attention_mask=b["attention_mask"]))[b["rows"], b["cols"]]
        sl = logits_of(student(input_ids=b["input_ids"],
                               attention_mask=b["attention_mask"]))[b["rows"], b["cols"]]
        loss, _, _ = hinton_kd_loss(sl, tl, b["labels"], 4.0, 0.95)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward(); scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(student.parameters(), 1.0)
    scaler.step(opt); scaler.update()
    if i == a.warmup - 1:                       # discard warmup: cudnn autotune,
        torch.cuda.synchronize()                # allocator growth, first kernels
        t0 = time.time()
        used = pad = 0
    elif t0 is not None:                        # elif: this batch's tokens are only
        used += int(b["attention_mask"].sum())  # counted if its time was too
        pad += int(b["input_ids"].numel())
torch.cuda.synchronize()
dt = time.time() - t0
print("TOKENS_PER_S %.1f" % (used / dt))
print("PAD_FRAC %.4f" % (1 - used / pad))
print("PEAK_GB %.2f" % (torch.cuda.max_memory_allocated() / 1e9))
""")

MAX_TOKENS = 8192
r = subprocess.run(["python", "-u", PROBE, "--subset", SUBSET, "--teacher", TEACH,
                    "--student", INIT, "--max-tokens", str(MAX_TOKENS)],
                   cwd=CODE, capture_output=True, text=True)
print(r.stdout[-1500:])
if r.returncode != 0:
    print("----- STDERR -----"); print(r.stderr[-2500:])
assert r.returncode == 0, "throughput probe failed"

vals = {}
for line in r.stdout.splitlines():
    parts = line.split()
    if len(parts) == 2 and parts[0] in ("TOKENS_PER_S", "PAD_FRAC", "PEAK_GB"):
        vals[parts[0]] = parts[1]
RATE = float(vals["TOKENS_PER_S"])
print("\nmeasured %.0f tokens/s | padding %.1f%% | peak %.2f GB"
      % (RATE, 100 * float(vals["PAD_FRAC"]), float(vals["PEAK_GB"])))
for ep in (1, 2, 3):
    print("   %d epoch(s): %5.1f h" % (ep, TOTAL_TOKENS * ep / RATE / 3600))
print("\nKaggle kills a session at 12 h. Pick EPOCHS in Cell 7 accordingly --")
print("the run resumes from its own checkpoint, so >12 h just means >1 session.")


In [ ]:

# -- Cell 7 -- launch. One GPU, resumable, mirrored to Drive.
#
# alpha=0.95 and T=4 make this PURE distillation: the effective weights are 0.05
# on the hard cross-entropy and 0.95 * 4^2 = 15.2 on the soft KL. The T^2 factor
# is Hinton's gradient-scale correction, not decoration -- softening by T shrinks
# the soft gradients by ~1/T^2, so without it alpha would not mean what it says.
#
# There is no control arm here. The comparison this run is meant to support is
# against the warm-start (their released 32M, untouched) and against the earlier
# cached-KD student, both of which already exist.
#
# STOP_HOURS stops the trainer before Kaggle's hard kill. It does not make the
# trainer save on the way out -- it buys time for the FINAL rclone sync to finish,
# so the newest on-disk checkpoint actually reaches Drive instead of being cut off
# mid-upload. Checkpoints land every 500 steps regardless, so the work at risk is
# at most those 500 steps either way.
EPOCHS = 2
STOP_HOURS = 11.0

OUT = WORK + "/runs/kd_live"
DEST = REMOTE + "/results/kd_live"
os.makedirs(OUT, exist_ok=True)

print("pulling any previous run state to resume...")
subprocess.run("rclone copy %s %s --transfers 8 -P" % (DEST, OUT), shell=True, check=False)
if os.path.exists(OUT + "/latest.pt"):
    print("found latest.pt -- this session will CONTINUE, not restart")

log_path = "/tmp/kd_live.log"
log = open(log_path, "w")
cmd = ["python", "-u", "train_kd.py",
       "--subset", SUBSET, "--teacher", TEACH, "--student", INIT, "--out", OUT,
       "--temperature", "4.0", "--alpha", "0.95",
       "--epochs", str(EPOCHS), "--max-tokens", str(MAX_TOKENS),
       "--lr", "1e-4", "--warmup", "2000",
       "--log-every", "200", "--save-every", "500", "--eval-every", "5000"]
proc = subprocess.Popen(cmd, cwd=CODE, stdout=log, stderr=subprocess.STDOUT,
                        env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"))
print("launched pid %d\n%s\n" % (proc.pid, " ".join(cmd)))

# The first thing train_kd.py does is count batches for the LR schedule, which
# tokenizes all 2M molecules. Expect several quiet minutes before step 1; the
# count is cached to nsteps.json and reused on every resume.
SYNC = subprocess.Popen(
    "while true; do rclone copy %s %s --drive-chunk-size 64M "
    ">> /tmp/rclone_sync.log 2>&1; sleep 600; done" % (OUT, DEST), shell=True)
print("background sync -> Drive every 10 min\n")

t0 = time.time()
while proc.poll() is None:
    time.sleep(300)
    hrs = (time.time() - t0) / 3600
    try:
        lines = [l.rstrip() for l in open(log_path) if l.startswith("[kd]") or "[val]" in l]
        last = lines[-1] if lines else "(counting batches / warming up)"
    except OSError:
        last = "(no log)"
    print("[%5.2f h] %s" % (hrs, last))
    if hrs >= STOP_HOURS:
        print("\nreached STOP_HOURS -- terminating cleanly so the checkpoint lands")
        proc.send_signal(signal.SIGINT)
        try:
            proc.wait(timeout=180)
        except subprocess.TimeoutExpired:
            proc.kill()
        break

SYNC.kill()
print("\ntrainer exit %s after %.2f h" % (proc.poll(), (time.time() - t0) / 3600))
print("".join(open(log_path).readlines()[-25:]))
subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (OUT, DEST),
               shell=True, check=True)
print("final sync done -> " + DEST)


In [ ]:

# -- Cell 8 -- curves, and what the numbers do and do not say.
#
# soft is KL(teacher || student) at T=4 -- the quantity actually being minimised.
# agree_teacher is the fraction of masked positions where the student's argmax
# matches the teacher's, which is the interpretable version of the same thing.
# It starts around 0.87 BEFORE any training, because the student is warm-started
# from their released 32M; only movement above that baseline is attributable to
# this run.
#
# None of this is evidence the student is better at anything downstream. The
# earlier cached-KD student improved on every intrinsic probe and still lost to
# its own starting point on PAMPA. Treat these curves as "did the optimisation do
# what it was told", nothing more.
import matplotlib.pyplot as plt

hist = json.load(open(OUT + "/history.json"))
tr = pd.DataFrame([h for h in hist if "loss" in h])
ev = pd.DataFrame([h["eval"] for h in hist if "eval" in h])
print("steps %d | epochs seen %s" % (len(tr), sorted(tr.epoch.unique())))
print(tr.tail(1).to_string(index=False))

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
w = tr.rolling(200, min_periods=1).mean()
ax[0].plot(tr.step, w.loss);  ax[0].set_title("total")
ax[1].plot(tr.step, w.soft);  ax[1].set_title("soft = KL(teacher || student), T=4")
ax[2].plot(tr.step, w.hard);  ax[2].set_title("hard = CE vs true token")
if len(ev):
    ax[1].plot(ev.step, ev.soft, "o-", label="held-out val"); ax[1].legend()
for x in ax:
    x.set_xlabel("step"); x.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT + "/curves.png", dpi=120); plt.show()

if len(ev):
    print("\nheld-out validation (20k molecules never trained on):")
    print(ev[["step", "total", "soft", "agree_teacher", "acc_student"]].to_string(index=False))

subprocess.run("rclone copy %s %s --drive-chunk-size 64M -P" % (OUT, DEST),
               shell=True, check=True)
print("\nuploaded to " + DEST)
print(subprocess.run("rclone lsf -R " + DEST, shell=True,
                     capture_output=True, text=True).stdout)
